# Tarea 1 - Notebook de análisis exploratorio de datos y Machine Learning básico

**Tema:** Análisis exploratorio y modelo básico de Machine Learning para la predicción del riesgo académico estudiantil en entornos virtuales de aprendizaje  
**Estudiante:** Milton Elías Espinoza Villalba  
**Módulo:** MTDI202: Big Data, Analytics & Data Scientist  
**Docente:** Ing. Carlos Wladimir Carrillo Villavicencio MSc. TIC.  
**Fecha:** Quito, junio de 2026

---

## Propósito del notebook

Este notebook desarrolla un proceso aplicado de análisis de datos con Python. Se utiliza el **Open University Learning Analytics Dataset (OULAD)** para integrar datos de estudiantes, cursos, evaluaciones e interacciones en un entorno virtual de aprendizaje. El objetivo analítico es identificar patrones asociados al rendimiento académico y construir un modelo básico de clasificación supervisada para predecir riesgo académico.

> Nota ética: el dataset es público y anonimizado. No se trabaja con datos personales identificables.

## Índice

1. Carga de librerías  
2. Descarga, carga y lectura del dataset  
3. Descripción inicial del dataset  
4. Limpieza y preparación de datos  
5. Integración de DataFrames y creación de variables  
6. Análisis exploratorio de datos (EDA)  
7. Visualizaciones e interpretación  
8. Preparación para Machine Learning  
9. Modelo supervisado con scikit-learn  
10. Evaluación de resultados  
11. Conclusiones y recomendaciones  
12. Checklist técnico de métodos aplicados

## 1. Carga de librerías

Se importan librerías para manejo de DataFrames, análisis numérico, visualización y Machine Learning supervisado. Las librerías principales son `pandas`, `numpy`, `matplotlib`, `seaborn` y `scikit-learn`.

In [ ]:
# Librerías básicas
import os
import zipfile
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, RocCurveDisplay
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)

print("Librerías cargadas correctamente.")

## 2. Descarga, carga y lectura del dataset

El dataset seleccionado es **Open University Learning Analytics Dataset (OULAD)**. El repositorio UCI lo describe como un conjunto de datos sobre cursos, estudiantes e interacciones con un entorno virtual de aprendizaje. Los datos están organizados en varios archivos CSV conectados por identificadores.

La celda siguiente intenta descargar el dataset desde UCI. Si la descarga falla, se debe descargar manualmente el archivo ZIP desde la página oficial de UCI u OULAD, subirlo a Colab y ejecutar nuevamente la extracción.

In [ ]:
# Carpeta de trabajo en Google Colab
BASE_DIR = Path("/content/oulad_project")
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = BASE_DIR / "oulad.zip"

# URL oficial de descarga desde UCI Machine Learning Repository
URLS = [
    "https://archive.ics.uci.edu/static/public/349/open+university+learning+analytics+dataset.zip",
    "https://archive.ics.uci.edu/static/public/349/open%2Buniversity%2Blearning%2Banalytics%2Bdataset.zip"
]

def download_file(urls, zip_path):
    if zip_path.exists() and zip_path.stat().st_size > 1_000_000:
        print(f"ZIP ya existe: {zip_path}")
        return True
    for url in urls:
        try:
            print(f"Intentando descargar desde: {url}")
            urllib.request.urlretrieve(url, zip_path)
            if zip_path.exists() and zip_path.stat().st_size > 1_000_000:
                print("Descarga completada.")
                return True
        except Exception as e:
            print(f"No se pudo descargar desde esa URL: {e}")
    return False

ok = download_file(URLS, ZIP_PATH)

if ok:
    try:
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
        print(f"Dataset extraído en: {DATA_DIR}")
    except Exception as e:
        print("El archivo descargado no pudo extraerse. Descargue el ZIP manualmente y súbalo a Colab.")
        print(e)
else:
    print("No fue posible descargar automáticamente el dataset.")
    print("Acción: descargue el ZIP desde UCI/OULAD, súbalo a Colab y colóquelo en /content/oulad_project/oulad.zip")

In [ ]:
# Función para localizar archivos dentro de la carpeta extraída

def find_csv(filename, base_dir=DATA_DIR):
    matches = list(base_dir.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"No se encontró {filename}. Revise si el ZIP fue extraído correctamente.")
    return matches[0]

expected_files = [
    "courses.csv", "assessments.csv", "studentAssessment.csv", "studentInfo.csv",
    "studentRegistration.csv", "studentVle.csv", "vle.csv"
]

for f in expected_files:
    try:
        print(f, "->", find_csv(f))
    except Exception as e:
        print(f, "-> NO ENCONTRADO")

### 2.1. Lectura de archivos CSV

Se leen los archivos principales. Para `studentVle.csv`, que es el archivo más grande, se cargan únicamente las columnas necesarias para el análisis: módulo, presentación, estudiante, fecha relativa y número de clics.

In [ ]:
# Lectura de tablas principales
courses = pd.read_csv(find_csv("courses.csv"))
assessments = pd.read_csv(find_csv("assessments.csv"))
student_info = pd.read_csv(find_csv("studentInfo.csv"))
student_registration = pd.read_csv(find_csv("studentRegistration.csv"))
student_assessment = pd.read_csv(find_csv("studentAssessment.csv"))
vle = pd.read_csv(find_csv("vle.csv"))

# studentVle es grande; se leen solo columnas relevantes
student_vle = pd.read_csv(
    find_csv("studentVle.csv"),
    usecols=["code_module", "code_presentation", "id_student", "date", "sum_click"],
    dtype={"code_module": "category", "code_presentation": "category", "id_student": "int32", "date": "int16", "sum_click": "int32"}
)

tables = {
    "courses": courses,
    "assessments": assessments,
    "student_info": student_info,
    "student_registration": student_registration,
    "student_assessment": student_assessment,
    "student_vle": student_vle,
    "vle": vle
}

for name, df_temp in tables.items():
    print(f"{name}: {df_temp.shape[0]:,} filas x {df_temp.shape[1]} columnas")

## 3. Descripción inicial del dataset

En esta fase se revisa la estructura general de los DataFrames: primeras filas, últimas filas, dimensiones, columnas, tipos de datos, valores faltantes y duplicados. Esta revisión permite decidir qué transformaciones son necesarias antes del análisis.

In [ ]:
# Primeros registros
student_info.head()

In [ ]:
# Últimos registros
student_info.tail()

In [ ]:
# Dimensiones de cada tabla
resumen_tablas = pd.DataFrame({
    "tabla": list(tables.keys()),
    "filas": [df.shape[0] for df in tables.values()],
    "columnas": [df.shape[1] for df in tables.values()]
})
resumen_tablas

In [ ]:
# Columnas por tabla
for name, df_temp in tables.items():
    print(f"\n{name.upper()}")
    print(list(df_temp.columns))

In [ ]:
# Información técnica del DataFrame principal de estudiantes
student_info.info()

In [ ]:
# Tipos de datos
student_info.dtypes

In [ ]:
# Estadística descriptiva de variables numéricas
student_info.describe()

In [ ]:
# Descripción de variables categóricas
student_info.describe(include="object")

## 4. Limpieza y preparación de datos

Se revisan valores faltantes, duplicados, consistencia de tipos de datos y nombres de columnas. En este dataset, las fechas son relativas al inicio de cada presentación del curso; por ello se manejan como variables numéricas temporales y no como fechas calendario.

In [ ]:
# Valores faltantes por tabla
missing_summary = []
for name, df_temp in tables.items():
    miss = df_temp.isnull().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    for col, n in miss.items():
        missing_summary.append({"tabla": name, "variable": col, "nulos": int(n), "porcentaje": round(n / len(df_temp) * 100, 2)})

missing_df = pd.DataFrame(missing_summary)
missing_df if not missing_df.empty else "No se detectaron valores faltantes."

In [ ]:
# Duplicados por tabla
duplicados = pd.DataFrame({
    "tabla": list(tables.keys()),
    "duplicados": [df.duplicated().sum() for df in tables.values()]
})
duplicados

In [ ]:
# Eliminación de duplicados si existieran
for name in tables:
    before = tables[name].shape[0]
    tables[name] = tables[name].drop_duplicates()
    after = tables[name].shape[0]
    print(f"{name}: {before - after} duplicados eliminados")

courses = tables["courses"]
assessments = tables["assessments"]
student_info = tables["student_info"]
student_registration = tables["student_registration"]
student_assessment = tables["student_assessment"]
student_vle = tables["student_vle"]
vle = tables["vle"]

In [ ]:
# Renombrar algunas columnas para facilitar lectura si se desea trabajar con nombres en español
# En este notebook se conservan nombres originales por trazabilidad con la documentación del dataset.
student_info.columns

In [ ]:
# Revisión de categorías principales
categorical_cols_info = ["gender", "region", "highest_education", "imd_band", "age_band", "disability", "final_result"]
for col in categorical_cols_info:
    print(f"\n{col}")
    print(student_info[col].value_counts(dropna=False))

## 5. Integración de DataFrames y creación de variables

El dataset OULAD está distribuido en varias tablas. Para construir un dataset analítico único, se integran fuentes mediante llaves comunes: `code_module`, `code_presentation` e `id_student`. También se crean variables derivadas, como total de clics, días activos, promedio de calificación y riesgo académico.

In [ ]:
keys = ["code_module", "code_presentation", "id_student"]

# Agregación de interacciones en el entorno virtual de aprendizaje
vle_agg = (
    student_vle
    .groupby(keys)
    .agg(
        total_clicks=("sum_click", "sum"),
        avg_clicks=("sum_click", "mean"),
        max_clicks_day=("sum_click", "max"),
        active_days=("date", "nunique"),
        first_activity_day=("date", "min"),
        last_activity_day=("date", "max")
    )
    .reset_index()
)

vle_agg.head()

In [ ]:
# Integración de evaluaciones con metadatos de evaluación
assessment_full = student_assessment.merge(
    assessments,
    on="id_assessment",
    how="left",
    suffixes=("_student", "_assessment")
)

# Variable de entrega tardía: fecha de entrega del estudiante mayor al deadline de la evaluación
assessment_full["late_submission"] = np.where(
    assessment_full["date"].notnull(),
    assessment_full["date_submitted"] > assessment_full["date"],
    False
)

assessment_agg = (
    assessment_full
    .groupby(keys)
    .agg(
        avg_score=("score", "mean"),
        min_score=("score", "min"),
        max_score=("score", "max"),
        n_assessments_submitted=("id_assessment", "count"),
        avg_days_submission=("date_submitted", "mean"),
        late_submissions=("late_submission", "sum"),
        banked_assessments=("is_banked", "sum")
    )
    .reset_index()
)

assessment_agg.head()

In [ ]:
# Construcción del dataset analítico integrado
student_dataset = (
    student_info
    .merge(student_registration, on=keys, how="left")
    .merge(courses, on=["code_module", "code_presentation"], how="left")
    .merge(vle_agg, on=keys, how="left")
    .merge(assessment_agg, on=keys, how="left")
)

# Variable objetivo: riesgo académico
student_dataset["riesgo_academico"] = student_dataset["final_result"].isin(["Withdrawn", "Fail"]).astype(int)
student_dataset["riesgo_academico_label"] = student_dataset["riesgo_academico"].map({1: "Riesgo académico", 0: "Sin riesgo académico"})

# Variable auxiliar: retiro registrado. No se usará como predictora para evitar fuga de información.
student_dataset["is_unregistered"] = student_dataset["date_unregistration"].notnull().astype(int)

# Tratamiento de nulos en variables creadas por integración
num_fill_zero = [
    "total_clicks", "avg_clicks", "max_clicks_day", "active_days", "first_activity_day", "last_activity_day",
    "avg_score", "min_score", "max_score", "n_assessments_submitted", "avg_days_submission",
    "late_submissions", "banked_assessments"
]
for col in num_fill_zero:
    if col in student_dataset.columns:
        student_dataset[col] = student_dataset[col].fillna(0)

# Tratamiento de nulos categóricos
student_dataset["imd_band"] = student_dataset["imd_band"].fillna("Unknown")

print(student_dataset.shape)
student_dataset.head()

In [ ]:
# Revisión final del dataset analítico
student_dataset.info()

In [ ]:
# Valores faltantes finales por columna
student_dataset.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
# Variables únicas por columna clave
student_dataset[["code_module", "code_presentation", "id_student", "final_result", "riesgo_academico_label"]].nunique()

## 6. Análisis exploratorio de datos (EDA)

El EDA permite comprender la estructura del dataset, identificar patrones, comparar grupos y detectar relaciones preliminares entre variables antes de entrenar el modelo. Se analizan variables categóricas, numéricas y temporales relativas.

In [ ]:
# Distribución de la variable objetivo
student_dataset["riesgo_academico_label"].value_counts()

In [ ]:
# Proporción de la variable objetivo
risk_rate = student_dataset["riesgo_academico"].mean()
print(f"Porcentaje de estudiantes en riesgo académico: {risk_rate:.2%}")

In [ ]:
# Estadística descriptiva de variables analíticas relevantes
selected_numeric = [
    "num_of_prev_attempts", "studied_credits", "date_registration", "module_presentation_length",
    "total_clicks", "avg_clicks", "active_days", "avg_score", "n_assessments_submitted", "late_submissions"
]
student_dataset[selected_numeric].describe().T

In [ ]:
# Agrupación por resultado final
result_summary = (
    student_dataset
    .groupby("final_result")
    .agg(
        estudiantes=("id_student", "count"),
        promedio_clicks=("total_clicks", "mean"),
        mediana_clicks=("total_clicks", "median"),
        promedio_score=("avg_score", "mean"),
        promedio_creditos=("studied_credits", "mean"),
        promedio_intentos_previos=("num_of_prev_attempts", "mean")
    )
    .round(2)
    .sort_values("estudiantes", ascending=False)
)
result_summary

In [ ]:
# Análisis por nivel educativo previo
education_risk = (
    student_dataset
    .groupby("highest_education")
    .agg(
        estudiantes=("id_student", "count"),
        tasa_riesgo=("riesgo_academico", "mean"),
        promedio_score=("avg_score", "mean"),
        promedio_clicks=("total_clicks", "mean")
    )
    .assign(tasa_riesgo_pct=lambda x: x["tasa_riesgo"] * 100)
    .sort_values("tasa_riesgo", ascending=False)
    .round(2)
)
education_risk

In [ ]:
# Tabla cruzada entre género y riesgo académico
pd.crosstab(
    student_dataset["gender"],
    student_dataset["riesgo_academico_label"],
    margins=True,
    normalize="index"
).round(3)

In [ ]:
# Tabla dinámica: promedio de clics por módulo y resultado final
pivot_clicks = pd.pivot_table(
    student_dataset,
    values="total_clicks",
    index="code_module",
    columns="final_result",
    aggfunc="mean"
).round(2)
pivot_clicks

In [ ]:
# Correlación entre variables numéricas seleccionadas
corr_matrix = student_dataset[selected_numeric + ["riesgo_academico"]].corr(numeric_only=True)
corr_matrix.round(3)

In [ ]:
# Filtro con query: estudiantes con alta interacción y buen promedio de evaluación
high_engagement = student_dataset.query("total_clicks > total_clicks.quantile(0.75) and avg_score >= 70")
print(f"Estudiantes con alta interacción y promedio >= 70: {high_engagement.shape[0]:,}")
high_engagement[["id_student", "code_module", "code_presentation", "total_clicks", "avg_score", "final_result"]].head()

In [ ]:
# Uso de loc / iloc para inspección específica
student_dataset.loc[:, ["id_student", "code_module", "code_presentation", "final_result", "riesgo_academico_label"]].iloc[:10]

## 7. Visualizaciones e interpretación

Cada gráfico incluye una interpretación. El objetivo no es solo generar visualizaciones, sino convertirlas en evidencia para responder la problemática analítica.

In [ ]:
# Configuración visual general
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
# Figura 1. Distribución del resultado final
ax = sns.countplot(data=student_dataset, x="final_result", order=student_dataset["final_result"].value_counts().index)
ax.set_title("Figura 1. Distribución del resultado final de los estudiantes")
ax.set_xlabel("Resultado final")
ax.set_ylabel("Número de estudiantes")
plt.xticks(rotation=0)
plt.show()

print("Interpretación: este gráfico permite identificar la composición general de resultados finales y observar el peso relativo de aprobación, reprobación, retiro y distinción.")

In [ ]:
# Figura 2. Proporción de riesgo académico
risk_counts = student_dataset["riesgo_academico_label"].value_counts()
plt.figure(figsize=(7, 7))
plt.pie(risk_counts.values, labels=risk_counts.index, autopct="%1.1f%%", startangle=90)
plt.title("Figura 2. Proporción de estudiantes según riesgo académico")
plt.show()

print(f"Interpretación: el {student_dataset['riesgo_academico'].mean():.1%} de los registros se clasifica como riesgo académico según la regla definida.")

In [ ]:
# Figura 3. Histograma de clics totales
sns.histplot(data=student_dataset, x="total_clicks", bins=50, kde=True)
plt.title("Figura 3. Distribución de clics totales en el entorno virtual")
plt.xlabel("Total de clics")
plt.ylabel("Frecuencia")
plt.show()

print("Interpretación: el histograma muestra la dispersión del nivel de interacción digital; normalmente se esperan asimetrías porque algunos estudiantes interactúan mucho más que otros.")

In [ ]:
# Figura 4. Boxplot de clics por resultado final
sns.boxplot(data=student_dataset, x="final_result", y="total_clicks")
plt.title("Figura 4. Clics totales por resultado final")
plt.xlabel("Resultado final")
plt.ylabel("Total de clics")
plt.yscale("symlog")
plt.show()

print("Interpretación: este boxplot permite comparar la interacción digital entre grupos de resultado final e identificar posibles valores atípicos.")

In [ ]:
# Figura 5. Tasa de riesgo por nivel educativo previo
plot_df = education_risk.reset_index()
sns.barplot(data=plot_df, y="highest_education", x="tasa_riesgo_pct")
plt.title("Figura 5. Tasa de riesgo académico por nivel educativo previo")
plt.xlabel("Tasa de riesgo académico (%)")
plt.ylabel("Nivel educativo previo")
plt.show()

print("Interpretación: la gráfica permite observar si ciertos niveles educativos previos presentan una tasa mayor de riesgo académico.")

In [ ]:
# Figura 6. Tasa de riesgo por banda de edad
age_risk = student_dataset.groupby("age_band").agg(tasa_riesgo=("riesgo_academico", "mean"), estudiantes=("id_student", "count")).reset_index()
age_risk["tasa_riesgo_pct"] = age_risk["tasa_riesgo"] * 100
sns.barplot(data=age_risk, x="age_band", y="tasa_riesgo_pct")
plt.title("Figura 6. Tasa de riesgo académico por banda de edad")
plt.xlabel("Banda de edad")
plt.ylabel("Tasa de riesgo académico (%)")
plt.show()

print("Interpretación: esta visualización permite comparar si la edad se asocia con diferencias en riesgo académico dentro del dataset.")

In [ ]:
# Figura 7. Violin plot del promedio de calificaciones por riesgo
sns.violinplot(data=student_dataset, x="riesgo_academico_label", y="avg_score", inner="quartile")
plt.title("Figura 7. Distribución del promedio de evaluación por riesgo académico")
plt.xlabel("Categoría")
plt.ylabel("Promedio de evaluación")
plt.show()

print("Interpretación: el gráfico permite comparar la distribución del promedio de evaluación entre estudiantes con riesgo y sin riesgo académico.")

In [ ]:
# Figura 8. Tendencia temporal relativa de clics por día del curso
clicks_by_day = student_vle.groupby("date", as_index=False).agg(total_clicks=("sum_click", "sum"))
sns.lineplot(data=clicks_by_day, x="date", y="total_clicks")
plt.title("Figura 8. Evolución temporal relativa de clics en la plataforma")
plt.xlabel("Día relativo al inicio del curso")
plt.ylabel("Total de clics")
plt.show()

print("Interpretación: la línea temporal permite observar patrones de actividad digital a lo largo del curso, incluyendo posibles picos vinculados a evaluaciones o semanas críticas.")

In [ ]:
# Figura 9. Mapa de calor de correlación
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="Blues", square=True)
plt.title("Figura 9. Mapa de calor de correlación entre variables numéricas")
plt.show()

print("Interpretación: el heatmap permite identificar relaciones lineales entre variables numéricas, incluyendo su asociación con la variable de riesgo académico.")

In [ ]:
# Figura 10. Dispersión entre clics y promedio de evaluación
sample_plot = student_dataset.sample(min(5000, len(student_dataset)), random_state=42)
sns.scatterplot(data=sample_plot, x="total_clicks", y="avg_score", hue="riesgo_academico_label", alpha=0.6)
plt.title("Figura 10. Relación entre clics totales y promedio de evaluación")
plt.xlabel("Total de clics")
plt.ylabel("Promedio de evaluación")
plt.xscale("symlog")
plt.legend(title="Categoría")
plt.show()

print("Interpretación: el gráfico permite explorar si una mayor interacción en la plataforma se relaciona visualmente con mejores calificaciones y menor riesgo académico.")

In [ ]:
# Figura 11. Barras agrupadas: módulo y resultado final
module_result = pd.crosstab(student_dataset["code_module"], student_dataset["final_result"], normalize="index")
module_result.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.title("Figura 11. Composición porcentual del resultado final por módulo")
plt.xlabel("Módulo")
plt.ylabel("Proporción")
plt.legend(title="Resultado final", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

print("Interpretación: la visualización muestra si la distribución de resultados finales varía entre módulos.")

## 8. Preparación para Machine Learning

Se define un problema de **clasificación supervisada**. La variable objetivo es `riesgo_academico`, construida a partir de `final_result`. Se excluyen identificadores y variables que pueden provocar fuga de información, como `date_unregistration`, porque conocer la fecha de retiro anticipa directamente una parte de la variable objetivo.

In [ ]:
# Selección de variables predictoras
features_categorical = [
    "code_module", "code_presentation", "gender", "region", "highest_education", "imd_band", "age_band", "disability"
]

features_numeric = [
    "num_of_prev_attempts", "studied_credits", "date_registration", "module_presentation_length",
    "total_clicks", "avg_clicks", "max_clicks_day", "active_days", "first_activity_day", "last_activity_day",
    "avg_score", "min_score", "max_score", "n_assessments_submitted", "avg_days_submission",
    "late_submissions", "banked_assessments"
]

# Dataset para ML
ml_df = student_dataset[features_categorical + features_numeric + ["riesgo_academico"]].copy()

# Asegurar tipos adecuados
for col in features_categorical:
    ml_df[col] = ml_df[col].astype("object").fillna("Unknown")

for col in features_numeric:
    ml_df[col] = pd.to_numeric(ml_df[col], errors="coerce")

X = ml_df[features_categorical + features_numeric]
y = ml_df["riesgo_academico"]

print("X:", X.shape)
print("y:", y.shape)
print("Distribución de y:")
print(y.value_counts(normalize=True).round(3))

In [ ]:
# Separación entrenamiento/prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

In [ ]:
# Preprocesamiento para variables numéricas y categóricas
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, features_numeric),
        ("cat", categorical_transformer, features_categorical)
    ]
)

## 9. Modelo supervisado con scikit-learn

Se entrenan dos modelos básicos:

1. **Regresión logística:** modelo base interpretable para clasificación binaria.  
2. **Random Forest Classifier:** modelo basado en ensamble de árboles, útil para capturar relaciones no lineales.

Ambos modelos se evalúan con métricas de clasificación: accuracy, precision, recall y F1-score.

In [ ]:
# Definición de modelos
models = {
    "Regresión logística": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=42, class_weight="balanced", n_jobs=-1)
}

results = []
trained_pipelines = {}

for model_name, model in models.items():
    clf = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    # Para AUC se requiere predict_proba si está disponible
    try:
        y_proba = clf.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_proba)
    except Exception:
        auc = np.nan

    results.append({
        "modelo": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": auc
    })

    trained_pipelines[model_name] = clf

results_df = pd.DataFrame(results).sort_values("f1_score", ascending=False)
results_df.round(4)

In [ ]:
# Selección automática del mejor modelo según F1-score
best_model_name = results_df.iloc[0]["modelo"]
best_model = trained_pipelines[best_model_name]
y_pred_best = best_model.predict(X_test)

print("Mejor modelo según F1-score:", best_model_name)
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_best, target_names=["Sin riesgo", "Riesgo"]))

## 10. Evaluación de resultados

Se presenta la matriz de confusión y, si el modelo lo permite, la curva ROC. La matriz de confusión es importante porque permite observar falsos positivos y falsos negativos. En un contexto educativo, los falsos negativos son críticos porque representan estudiantes en riesgo que el modelo no detecta.

In [ ]:
# Figura 12. Matriz de confusión
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Sin riesgo", "Riesgo"])
disp.plot(values_format="d")
plt.title(f"Figura 12. Matriz de confusión - {best_model_name}")
plt.show()

print("Interpretación: la matriz permite identificar aciertos y errores del modelo. En educación, debe prestarse atención especial a los falsos negativos, porque implican estudiantes en riesgo no detectados.")

In [ ]:
# Figura 13. Curva ROC del mejor modelo, si aplica
try:
    y_score = best_model.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, y_score)
    plt.title(f"Figura 13. Curva ROC - {best_model_name}")
    plt.show()
    print(f"AUC ROC del mejor modelo: {roc_auc_score(y_test, y_score):.4f}")
except Exception as e:
    print("El modelo seleccionado no permite generar curva ROC con predict_proba.")

In [ ]:
# Optimización básica opcional con GridSearchCV para Random Forest
# Esta celda puede tardar varios minutos. Si el tiempo es limitado, puede dejarse ejecutada solo una vez.

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [8, 12, None],
    "model__min_samples_leaf": [1, 3]
}

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42, class_weight="balanced", n_jobs=-1))
])

grid = GridSearchCV(
    rf_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)
print("Mejores parámetros:", grid.best_params_)
print("Mejor F1 en validación cruzada:", round(grid.best_score_, 4))

optimized_model = grid.best_estimator_
y_pred_opt = optimized_model.predict(X_test)

optimized_results = {
    "accuracy": accuracy_score(y_test, y_pred_opt),
    "precision": precision_score(y_test, y_pred_opt, zero_division=0),
    "recall": recall_score(y_test, y_pred_opt, zero_division=0),
    "f1_score": f1_score(y_test, y_pred_opt, zero_division=0)
}
optimized_results

In [ ]:
# Importancia de variables para Random Forest optimizado o modelo Random Forest base
# La importancia después de OneHotEncoder requiere recuperar nombres finales de variables.
try:
    model_for_importance = optimized_model
except NameError:
    model_for_importance = trained_pipelines["Random Forest"]

try:
    feature_names_num = features_numeric
    feature_names_cat = list(model_for_importance.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(features_categorical))
    feature_names = feature_names_num + feature_names_cat
    importances = model_for_importance.named_steps["model"].feature_importances_
    importance_df = pd.DataFrame({"variable": feature_names, "importancia": importances}).sort_values("importancia", ascending=False).head(20)
    display(importance_df)

    sns.barplot(data=importance_df, x="importancia", y="variable")
    plt.title("Figura 14. Principales variables según importancia del modelo Random Forest")
    plt.xlabel("Importancia")
    plt.ylabel("Variable")
    plt.show()
except Exception as e:
    print("No se pudo calcular importancia de variables:", e)

## 11. Conclusiones y recomendaciones

Las conclusiones deben actualizarse después de ejecutar el notebook completo. A partir del análisis, se recomienda redactar conclusiones que conecten: problemática, EDA, visualizaciones y desempeño del modelo.

### Conclusiones preliminares

1. El dataset OULAD permite analizar el riesgo académico porque integra información demográfica, académica, evaluativa e interacción digital en un entorno virtual de aprendizaje.  
2. La variable `riesgo_academico` permite convertir el problema educativo en una tarea de clasificación supervisada.  
3. Las variables de interacción en la plataforma y desempeño en evaluaciones pueden aportar señales útiles para identificar estudiantes con mayor probabilidad de retiro o reprobación.  
4. El modelo debe interpretarse como una herramienta de apoyo analítico y no como sustituto del juicio docente o de la intervención pedagógica.  

### Recomendaciones preliminares

1. Utilizar el modelo como mecanismo de alerta temprana, complementándolo con criterios pedagógicos y tutorías personalizadas.  
2. Evitar decisiones automáticas sobre estudiantes; las predicciones deben servir para orientar apoyo, no para sancionar.  
3. Mejorar el análisis futuro usando ventanas temporales tempranas, por ejemplo clics y evaluaciones de las primeras semanas del curso.  
4. Comparar más modelos y revisar métricas sensibles al riesgo, especialmente recall, porque interesa detectar correctamente a estudiantes vulnerables.

## 12. Checklist técnico de métodos aplicados

La siguiente tabla evidencia el uso de operaciones y técnicas de manejo de DataFrames requeridas por la tarea. No se aplican de manera mecánica, sino con sentido analítico.

In [ ]:
checklist_metodos = pd.DataFrame({
    "Método/técnica": [
        "head()", "tail()", "shape", "columns", "info()", "dtypes", "describe()", "describe(include='object')",
        "isnull().sum()", "duplicated().sum()", "drop_duplicates()", "fillna()", "astype()", "value_counts()",
        "nunique()", "sort_values()", "loc", "iloc", "query()", "groupby()", "agg()", "pd.crosstab()",
        "pivot_table()", "assign()", "corr()", "merge()", "apply/lambda", "train_test_split", "fit()", "predict()"
    ],
    "Uso en el notebook": [
        "Vista inicial de registros", "Vista final de registros", "Dimensión de DataFrames", "Revisión de nombres de columnas",
        "Estructura, nulos y tipos", "Confirmación de tipos", "Estadística descriptiva numérica", "Resumen de categóricas",
        "Detección de faltantes", "Detección de duplicados", "Eliminación justificada de duplicados", "Tratamiento de nulos",
        "Conversión de tipos", "Frecuencias categóricas", "Conteo de valores únicos", "Ordenamiento de tablas resumen",
        "Selección de columnas", "Selección por posición", "Filtro analítico", "Agregación por grupos", "Métricas agregadas",
        "Cruce de variables categóricas", "Tabla dinámica analítica", "Creación ordenada de variables", "Correlaciones",
        "Integración de tablas", "Transformaciones personalizadas", "División de datos", "Entrenamiento del modelo", "Predicción del modelo"
    ]
})
checklist_metodos

## Cierre del notebook

Después de ejecutar todas las celdas, revise:

- Que no existan errores de ejecución.
- Que las visualizaciones se muestren correctamente.
- Que el modelo entregue métricas de evaluación.
- Que las conclusiones del informe coincidan con los resultados obtenidos.
- Que el repositorio de GitHub esté público y tenga el enlace funcional en los anexos del informe.